In [3]:
!pip install beautifulsoup4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.9/147.9 kB 1.8 MB/s eta 0:00:00a 0:00:01m

[notice] A new release of pip is available: 24.0 -> 24.1.2
[notice] To update, run: pip install --upgrade pip


In [17]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# URL of the economic calendar page on Investing.com
url = "https://www.investing.com/economic-calendar/"

# Headers to mimic a real browser request
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Send a request to the website
response = requests.get(url)

# Check if the request was successful
if response.status_code == 200:
    # Parse the HTML content using BeautifulSoup
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Find the table or container holding the economic events
    calendar_table = soup.find('table', {'id': 'economicCalendarData'})
    
    # Initialize lists to store the scraped data
    dates = []
    times = []
    currencies = []
    events = []
    impacts = []
    actuals = []
    forecasts = []
    previous = []
    
    # Loop through the rows of the calendar table
    for row in calendar_table.find_all('tr', {'class': 'js-event-item'}):
        # Extract the event's priority (impact)
        impact = row.find('td', {'class': 'sentiment'}).get('title')
        
        # Only proceed if the event has a three-star impact
        if impact in ['Low Volatility Expected', 'Moderate Volatility Expected']:
        # if impact == 'High Volatility Expected':
            # Extract the other event details
            date = row.get('data-event-datetime')
            time = row.find('td', {'class': 'time'}).text.strip()
            currency = row.find('td', {'class': 'left flagCur noWrap'}).text.strip()
            event = row.find('td', {'class': 'left event'}).text.strip()
            actual = row.find('td', {'class': 'act'}).text.strip()
            forecast = row.find('td', {'class': 'fore'}).text.strip()
            prev = row.find('td', {'class': 'prev'}).text.strip()
            
            # Append the details to the respective lists
            dates.append(date)
            times.append(time)
            currencies.append(currency)
            events.append(event)
            impacts.append(impact)
            actuals.append(actual)
            forecasts.append(forecast)
            previous.append(prev)
    
    # Create a DataFrame from the lists
    data = {
        'Date': dates,
        'Time': times,
        'Currency': currencies,
        'Event': events,
        'Impact': impacts,
        'Actual': actuals,
        'Forecast': forecasts,
        'Previous': previous
    }
    df = pd.DataFrame(data)
else:
    print(f"Failed to retrieve data. HTTP Status code: {response.status_code}")


In [18]:
df

,Date,Time,Currency,Event,Impact,Actual,Forecast,Previous
0,2024/07/26 01:00:00,01:00,JPY,Coincident Indicator (MoM) (May),Low Volatility Expected,1.9%,1.3%,1.0%
1,2024/07/26 01:00:00,01:00,JPY,Leading Index (May),Low Volatility Expected,111.2,111.1,110.9
2,2024/07/26 01:00:00,01:00,JPY,Leading Index (MoM) (May),Low Volatility Expected,0.3%,0.2%,-0.8%
3,2024/07/26 01:00:00,01:00,SGD,Industrial Production (YoY) (Jun),Low Volatility Expected,-3.9%,0.0%,2.3%
4,2024/07/26 01:00:00,01:00,SGD,Industrial Production (MoM) (Jun),Low Volatility Expected,-3.8%,-0.5%,0.5%
5,2024/07/26 02:45:00,02:45,EUR,French Consumer Confidence (Jul),Low Volatility Expected,91,90,90
6,2024/07/26 03:00:00,03:00,EUR,Spanish Retail Sales (YoY) (Jun),Low Volatility Expected,0.3%,,0.2%
7,2024/07/26 03:00:00,03:00,EUR,Spanish Unemployment Rate (Q2),Low Volatility Expected,11.27%,11.40%,12.29%
8,2024/07/26 04:00:00,04:00,GBP,BoE Quarterly Bulletin,Low Volatility Expected,,,
9,2024/07/26 04:00:00,04:00,EUR,Italian Business Confidence (Jul),Low Volatility Expected,87.6,87.0,86.9


In [26]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# URL of the economic calendar page on Investing.com
url = "https://www.investing.com/economic-calendar/"

# Headers to mimic a real browser request
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Countries to filter (use the country abbreviations or names as they appear on the site)
countries_of_interest = ['United States', 'Germany', 'United Kingdom']  # Example countries

# Send a request to the website
response = requests.get(url, headers=headers)

# Check if the request was successful
if response.status_code == 200:
    # Parse the HTML content using BeautifulSoup
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Find the table or container holding the economic events
    calendar_table = soup.find('table', {'id': 'economicCalendarData'})
    
    # Initialize lists to store the scraped data
    dates = []
    times = []
    currencies = []
    events = []
    impacts = []
    actuals = []
    forecasts = []
    previous = []
    
    # Loop through the rows of the calendar table
    for row in calendar_table.find_all('tr', {'class': 'js-event-item'}):
        # Extract the event's priority (impact)
        impact = row.find('td', {'class': 'sentiment'}).get('title')
        
        # Extract the country (currency) information
        country = row.find('td', {'class': 'left flagCur noWrap'}).text.strip()
        
        # Only proceed if the event has a one or two-star impact and is from a country of interest
        if impact in ['Low Volatility Expected', 'Moderate Volatility Expected'] and country in countries_of_interest:
            # Extract the other event details
            date = row.get('data-event-datetime')
            time = row.find('td', {'class': 'time'}).text.strip()
            event = row.find('td', {'class': 'left event'}).text.strip()
            actual = row.find('td', {'class': 'act'}).text.strip()
            forecast = row.find('td', {'class': 'fore'}).text.strip()
            prev = row.find('td', {'class': 'prev'}).text.strip()
            
            # Append the details to the respective lists
            dates.append(date)
            times.append(time)
            currencies.append(country)
            events.append(event)
            impacts.append(impact)
            actuals.append(actual)
            forecasts.append(forecast)
            previous.append(prev)
    
    # Create a DataFrame from the lists
    data = {
        'Date': dates,
        'Time': times,
        'Country': currencies,
        'Event': events,
        'Impact': impacts,
        'Actual': actuals,
        'Forecast': forecasts,
        'Previous': previous
    }
    df = pd.DataFrame(data)
    
    # Save the DataFrame to a CSV file
    # df.to_csv('economic_calendar_filtered.csv', index=False)
    
    # Display the DataFrame
else:
    print(f"Failed to retrieve data. HTTP Status code: {response.status_code}")


In [25]:
df

,Date,Time,Country,Event,Impact,Actual,Forecast,Previous
